# 02 — Data Cleaning & Harmonization

**Project:** Causal ML for Climate Policy Effectiveness  
**Purpose:** Clean individual datasets, harmonize country identifiers, standardize time dimensions  
**Input:** `data/raw/` (from Notebook 01)  
**Output:** `data/interim/` — cleaned, harmonized datasets ready for merging

---

## Cleaning Steps

1. Clean OWID CO₂ data — filter countries, validate codes, select relevant columns  
2. Clean OWID Energy — extract energy mix confounders  
3. Clean Climate Policy Database — parse types, standardize sectors  
4. Clean World Bank indicators — interpolate gaps, validate ranges  
5. Harmonize all datasets on ISO 3166-1 alpha-3 codes  
6. Export cleaned datasets to `data/interim/`

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    clean_owid_co2, clean_owid_energy, clean_sector_emissions,
    harmonize_country_codes, interpolate_panel_gaps,
    NON_COUNTRY_ENTITIES
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

%load_ext watermark
%watermark -v -p pandas,numpy

Python implementation: CPython
Python version       : 3.10.11
IPython version      : 8.39.0

pandas: 2.3.3
numpy : 2.0.2



---
## 2.1 — Clean OWID CO₂ Data

In [2]:
print("=" * 60)
print("2.1 — Cleaning OWID CO₂ Dataset")
print("=" * 60)

df_co2_raw = pd.read_csv(RAW_DIR / "owid" / "owid-co2-data.csv")
print(f"Raw: {df_co2_raw.shape[0]:,} rows, {df_co2_raw.shape[1]} columns")

df_co2 = clean_owid_co2(df_co2_raw)
print(f"Cleaned: {df_co2.shape[0]:,} rows, {df_co2.shape[1]} columns")
print(f"\nMissing values (%):\n{(df_co2.isnull().mean() * 100).round(1).to_string()}")

2.1 — Cleaning OWID CO₂ Dataset


Raw: 50,411 rows, 79 columns
Cleaned: 7,630 rows, 15 columns

Missing values (%):
iso3                 0.0
country              0.0
year                 0.0
co2                  1.6
co2_per_capita       2.5
co2_per_gdp         29.1
coal_co2            37.7
oil_co2              2.3
gas_co2             44.2
cement_co2           4.8
co2_growth_prct      2.4
co2_growth_abs       1.8
share_global_co2     1.6
population           0.9
gdp                 29.1


In [3]:
# Validate: top emitters should be present
top_emitters = ["CHN", "USA", "IND", "RUS", "JPN", "DEU", "GBR", "BRA", "FRA", "CAN"]
present = [c for c in top_emitters if c in df_co2["iso3"].values]
missing = [c for c in top_emitters if c not in df_co2["iso3"].values]
print(f"Top emitters present: {len(present)}/{len(top_emitters)}")
if missing:
    print(f" Missing: {missing}")

df_co2.head()

Top emitters present: 10/10


,iso3,country,year,co2,co2_per_capita,co2_per_gdp,coal_co2,oil_co2,gas_co2,cement_co2,co2_growth_prct,co2_growth_abs,share_global_co2,population,gdp
0,ABW,Aruba,1990,0.487,7.561,NaN,NaN,0.487,NaN,0.0,-24.859,-0.161,0.002,64452.0,NaN
1,ABW,Aruba,1991,0.531,7.981,NaN,NaN,0.531,NaN,0.0,9.023,0.044,0.002,66564.0,NaN
2,ABW,Aruba,1992,0.539,7.818,NaN,NaN,0.539,NaN,0.0,1.379,0.007,0.002,68889.0,NaN
3,ABW,Aruba,1993,0.649,9.123,NaN,NaN,0.649,NaN,0.0,20.408,0.110,0.003,71091.0,NaN
4,ABW,Aruba,1994,0.660,8.976,NaN,NaN,0.660,NaN,0.0,1.695,0.011,0.003,73475.0,NaN


---
## 2.2 — Clean OWID Energy Data

In [4]:
print("=" * 60)
print("2.2 — Cleaning OWID Energy Dataset")
print("=" * 60)

df_energy_raw = pd.read_csv(RAW_DIR / "owid" / "owid-energy-data.csv")
print(f"Raw: {df_energy_raw.shape[0]:,} rows")

df_energy = clean_owid_energy(df_energy_raw)
print(f"Cleaned: {df_energy.shape[0]:,} rows, {df_energy.shape[1]} columns")
print(f"\nKey energy columns: {[c for c in df_energy.columns if 'share' in c or 'per_capita' in c]}")

2.2 — Cleaning OWID Energy Dataset


Raw: 23,377 rows
Cleaned: 7,696 rows, 15 columns

Key energy columns: ['renewables_share_energy', 'fossil_share_energy', 'renewables_share_elec', 'fossil_share_elec', 'solar_share_elec', 'wind_share_elec', 'hydro_share_elec', 'nuclear_share_elec', 'energy_per_capita']


---
## 2.3 — Clean Climate Policy Database

In [5]:
print("=" * 60)
print("2.3 — Cleaning Climate Policy Database")
print("=" * 60)

df_policies_raw = pd.read_csv(RAW_DIR / "climate_policy_db" / "climate_policy_database_policies_export.csv", dtype=str)
print(f"Raw: {df_policies_raw.shape[0]} policies")

df_policies = df_policies_raw.copy()

# Ensure we have the required columns for downstream merge
if 'country_iso' in df_policies.columns:
    df_policies['iso3'] = df_policies['country_iso']
else:
    df_policies = harmonize_country_codes(df_policies, "country")

if 'decision_date' in df_policies.columns:
    df_policies['year_adopted'] = pd.to_numeric(df_policies['decision_date'], errors='coerce')
    
# Drop NA years
df_policies = df_policies.dropna(subset=['year_adopted'])
df_policies['year_adopted'] = df_policies['year_adopted'].astype(int)

# The Climate Policy Database includes some entries with a future decision_date
# (e.g. 2026) -- these are announced/planned policies recorded ahead of the panel's
# actual coverage window (1990-2025), not a parsing artifact. Since a policy can't
# have caused emissions changes before it existed, drop entries whose adoption year
# falls after the panel's end year to avoid future-dated treatment adoption.
PANEL_END_YEAR = 2025
n_before_future_filter = len(df_policies)
df_policies = df_policies[df_policies['year_adopted'] <= PANEL_END_YEAR]
n_dropped_future = n_before_future_filter - len(df_policies)
if n_dropped_future > 0:
    print(f"Dropped {n_dropped_future} polic(y/ies) with adoption year after {PANEL_END_YEAR} "
          f"(announced/planned policies outside the panel's coverage window)")

# Categorize policies based on the policy_instrument column.
#
# NOTE ON A PRIOR BUG (found via external peer review): an earlier version of
# this function matched on bare substrings "tax", "trading", "allowance",
# "efficiency", "standard" -- each far too broad. "tax" alone matched generic
# instruments like "Tax relief" (a tax *reduction*, the opposite of a carbon
# price) and "Energy and other taxes" (not necessarily carbon-related at all).
# Concretely, this miscategorized India's 1944 "Basic Excise Duty" -- a
# colonial-era general excise tax with no connection to carbon or climate
# policy -- as "Carbon Pricing", and did the same to roughly a third of all
# "Carbon Pricing"-labeled records (146 "Tax relief" + 100+ generic "Energy
# and other taxes" entries out of 910). "efficiency"/"standard" similarly
# over-matched into "Efficiency Standard" (2,664 records under the old rule
# vs. 511 under the precise one below). Fixed by matching on specific,
# carbon/instrument-type-qualified phrases only.
import re

CARBON_PRICING_PATTERN = re.compile(
    r"co2 tax|carbon tax|carbon pricing|carbon price|"
    r"emissions? trading|cap.and.trade|ghg emissions allowance|"
    r"ghg emission reduction crediting",
    re.IGNORECASE,
)
RENEWABLE_MANDATE_PATTERN = re.compile(
    r"renewable energy target|feed-in tariff|feed in premium|"
    r"renewable mandate|renewable portfolio|net metering",
    re.IGNORECASE,
)
EFFICIENCY_STANDARD_PATTERN = re.compile(
    r"energy efficiency|efficiency standard|fuel-economy|fuel economy|"
    r"appliance standard|building code|minimum energy performance",
    re.IGNORECASE,
)


def categorize_policy(instrument_str):
    if not isinstance(instrument_str, str):
        return "Other"
    if CARBON_PRICING_PATTERN.search(instrument_str):
        return "Carbon Pricing"
    elif RENEWABLE_MANDATE_PATTERN.search(instrument_str):
        return "Renewable Mandate"
    elif EFFICIENCY_STANDARD_PATTERN.search(instrument_str):
        return "Efficiency Standard"
    else:
        return "Other"

if 'policy_instrument' in df_policies.columns:
    df_policies["policy_category"] = df_policies["policy_instrument"].apply(categorize_policy)
elif 'policy_type' in df_policies.columns:
    df_policies["policy_category"] = df_policies["policy_type"].apply(categorize_policy)
else:
    df_policies["policy_category"] = "Other"

# Handle EU policies — map to individual EU member states
eu_members = [
    "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN", "FRA",
    "DEU", "GRC", "HUN", "IRL", "ITA", "LVA", "LTU", "LUX", "MLT", "NLD",
    "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE"
]

eu_mask = df_policies["iso3"].isin(["EUU", "EUR"]) | (df_policies["country"] == "European Union")
eu_policies = df_policies[eu_mask].copy()

if len(eu_policies) > 0:
    expanded = []
    for _, row in eu_policies.iterrows():
        for member in eu_members:
            new_row = row.copy()
            new_row["iso3"] = member
            new_row["country"] = member
            expanded.append(new_row)
    df_eu_expanded = pd.DataFrame(expanded)
    df_policies = pd.concat([
        df_policies[~eu_mask],
        df_eu_expanded
    ], ignore_index=True)
    print(f"Expanded EU policies to {len(eu_members)} member states")

print(f"Cleaned: {len(df_policies)} policy entries")
print(f"\nPolicy categories:\n{df_policies['policy_category'].value_counts().to_string()}")
print(f"\nYear range: {df_policies['year_adopted'].min()} — {df_policies['year_adopted'].max()}")
print(f"Countries covered: {df_policies['iso3'].nunique()}")

# Sanity check: earliest Carbon Pricing record should be a real carbon-pricing
# instrument, not a generic historical tax (this is exactly the check that
# would have caught the original bug).
cp_check = df_policies[df_policies["policy_category"] == "Carbon Pricing"].sort_values("year_adopted")
print(f"\nEarliest Carbon Pricing record: {cp_check.iloc[0]['policy_name']} ({cp_check.iloc[0]['year_adopted']})")

2.3 — Cleaning Climate Policy Database


Raw: 6727 policies
Dropped 7 polic(y/ies) with adoption year after 2025 (announced/planned policies outside the panel's coverage window)


Expanded EU policies to 27 member states
Cleaned: 13134 policy entries

Policy categories:
policy_category
Other                  10689
Efficiency Standard     1000
Renewable Mandate        921
Carbon Pricing           524

Year range: 1927 — 2025
Countries covered: 197

Earliest Carbon Pricing record: Special Tax Law on Production and Services (including fuel tax) Mexico (1980) (1980)


---
## 2.3b — Clean Sector-Level CO2 Emissions

Loads `data/raw/wdi/wdi_sector_emissions.csv` -- a country x year panel of CO2
emissions disaggregated by sector (Power, Transport, Industry, Agriculture),
1990-2023. This file previously existed on disk but was never wired into the
pipeline; it now backs the sector-level DML/Causal Forest analysis (Notebook 05).

In [6]:
print("=" * 60)
print("2.3b - Cleaning Sector-Level CO2 Emissions")
print("=" * 60)

sector_path = RAW_DIR / "wdi" / "wdi_sector_emissions.csv"

if sector_path.exists():
    df_sector_raw = pd.read_csv(sector_path)
    print(f"Raw: {df_sector_raw.shape[0]:,} rows, {df_sector_raw.shape[1]} columns")
    df_sector = clean_sector_emissions(df_sector_raw)
    print(f"Cleaned: {df_sector.shape[0]:,} rows, {df_sector.shape[1]} columns")
    print(f"Countries: {df_sector['iso3'].nunique()}, years {df_sector['year'].min()}-{df_sector['year'].max()}")
else:
    print("  Sector emissions file not found -- sector-level analysis will be unavailable")
    df_sector = None

2.3b - Cleaning Sector-Level CO2 Emissions
Raw: 8,228 rows, 6 columns
Cleaned: 8,228 rows, 6 columns
Countries: 242, years 1990-2023


---
## 2.4 — Clean World Bank Indicators

In [7]:
print("=" * 60)
print("2.4 — Cleaning World Bank Indicators")
print("=" * 60)

wb_path = RAW_DIR / "world_bank" / "wb_indicators.csv"
wgi_path = RAW_DIR / "world_bank" / "wgidataset_with_sourcedata-2025.xlsx"

# World Bank Development Indicators
if wb_path.exists():
    df_wb = pd.read_csv(wb_path)
    print(f"WB Indicators: {df_wb.shape[0]:,} rows, {df_wb.shape[1]} columns")
    
    # Interpolate small gaps (max 3 consecutive years)
    import numpy as np
    numeric_cols = df_wb.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != "year"]
    # WDI is annual (unlike WGI's biennial-pre-2002 cadence), so a slightly
    # longer max_gap is defensible for smoothing short annual reporting gaps.
    
    df_wb = interpolate_panel_gaps(df_wb, group_col="iso3", time_col="year",
                                    value_cols=numeric_cols, max_gap=3)
    
    missing_pct = (df_wb[numeric_cols].isnull().mean() * 100).round(1)
    print(f"\nMissing after interpolation:\n{missing_pct.to_string()}")
else:
    print(" WB indicators not found — will use OWID GDP/population data")
    df_wb = None

# Governance Indicators
if wgi_path.exists():
    print(f"\nReading WGI Excel File: {wgi_path.name}...")
    wgi_map = {
        'rq': 'regulatory_quality',
        'rl': 'rule_of_law',
        'ge': 'govt_effectiveness',
        'cc': 'control_corruption'
    }
    dfs = []
    for sheet, col_name in wgi_map.items():
        tmp = pd.read_excel(wgi_path, sheet_name=sheet)
        tmp = tmp[['Economy (code)', 'Year', 'Governance estimate (approx. -2.5 to +2.5)']]
        tmp.columns = ['iso3', 'year', col_name]
        dfs.append(tmp)
    
    df_wgi = dfs[0]
    for df in dfs[1:]:
        df_wgi = pd.merge(df_wgi, df, on=['iso3', 'year'], how='outer')
    
    # WGI data is sometimes missing for certain years/countries, drop full NAs
    df_wgi = df_wgi.dropna(subset=list(wgi_map.values()), how='all')
    print(f"WGI Governance: {df_wgi.shape[0]:,} rows")
    
    # WGI is biennial before 2002, annual after — interpolate
    gov_cols = [c for c in df_wgi.columns if c not in ["iso3", "year"]]
    df_wgi = interpolate_panel_gaps(df_wgi, group_col="iso3", time_col="year",
                                     value_cols=gov_cols, max_gap=2)
    print(f"WGI countries: {df_wgi['iso3'].nunique()}")
else:
    print(" WGI data not found — governance moderators will be unavailable")
    df_wgi = None


2.4 — Cleaning World Bank Indicators
WB Indicators: 9,310 rows, 8 columns



Missing after interpolation:
energy_use_per_capita    24.2
trade_pct_gdp            20.8
industry_pct_gdp         13.5
gdp_per_capita            5.3
population                0.4
urban_pct                 0.4

Reading WGI Excel File: wgidataset_with_sourcedata-2025.xlsx...


WGI Governance: 5,477 rows


WGI countries: 216


---
## 2.5 — Merge Covariates

In [8]:
print("=" * 60)
print("2.5 — Merging Covariates")
print("=" * 60)

# Start with CO₂ emissions as the backbone
df_merged = df_co2.copy()
print(f"Starting panel: {df_merged.shape}")

# Merge energy data
energy_merge_cols = [c for c in df_energy.columns if c not in ["country"]]
df_merged = df_merged.merge(
    df_energy[energy_merge_cols],
    on=["iso3", "year"],
    how="left",
    suffixes=("", "_energy")
)
print(f"After energy merge: {df_merged.shape}")

# Merge World Bank indicators
if df_wb is not None:
    df_merged = df_merged.merge(
        df_wb, on=["iso3", "year"], how="left", suffixes=("", "_wb")
    )
    print(f"After WB merge: {df_merged.shape}")

# Merge governance indicators
if df_wgi is not None:
    df_merged = df_merged.merge(
        df_wgi, on=["iso3", "year"], how="left", suffixes=("", "_wgi")
    )
    print(f"After WGI merge: {df_merged.shape}")

# Merge sector-level CO2 emissions (Power/Transport/Industry/Agriculture)
if df_sector is not None:
    df_merged = df_merged.merge(
        df_sector, on=["iso3", "year"], how="left", suffixes=("", "_sector")
    )
    print(f"After sector emissions merge: {df_merged.shape}")

print(f"\nFinal merged panel: {df_merged.shape[0]:,} rows × {df_merged.shape[1]} columns")
print(f"Countries: {df_merged['iso3'].nunique()}")
print(f"Years: {df_merged['year'].min()} – {df_merged['year'].max()}")

2.5 — Merging Covariates
Starting panel: (7630, 15)
After energy merge: (7630, 27)
After WB merge: (7630, 33)
After WGI merge: (7630, 37)
After sector emissions merge: (7630, 41)

Final merged panel: 7,630 rows × 41 columns
Countries: 218
Years: 1990 – 2024


---
## 2.6 — Export Cleaned Datasets

In [9]:
print("=" * 60)
print("2.6 — Exporting Cleaned Datasets")
print("=" * 60)

# Save individual cleaned datasets
df_co2.to_csv(INTERIM_DIR / "emissions_cleaned.csv", index=False)
print(f"   emissions_cleaned.csv ({len(df_co2):,} rows)")

df_energy.to_csv(INTERIM_DIR / "energy_cleaned.csv", index=False)
print(f"   energy_cleaned.csv ({len(df_energy):,} rows)")

df_policies.to_csv(INTERIM_DIR / "policies_cleaned.csv", index=False)
print(f"   policies_cleaned.csv ({len(df_policies)} policies)")

# Save merged covariates
df_merged.to_csv(INTERIM_DIR / "covariates_merged.csv", index=False)
print(f"   covariates_merged.csv ({len(df_merged):,} rows)")

if df_wb is not None:
    df_wb.to_csv(INTERIM_DIR / "wb_indicators_cleaned.csv", index=False)
    print(f"   wb_indicators_cleaned.csv")

if df_wgi is not None:
    df_wgi.to_csv(INTERIM_DIR / "wgi_governance_cleaned.csv", index=False)
    print(f"   wgi_governance_cleaned.csv")

if df_sector is not None:
    df_sector.to_csv(INTERIM_DIR / "sector_emissions_cleaned.csv", index=False)
    print(f"   sector_emissions_cleaned.csv ({len(df_sector):,} rows)")

print(f"\n All cleaned datasets exported to: {INTERIM_DIR}")
print("  Proceed to Notebook 03: Exploratory Data Analysis")

2.6 — Exporting Cleaned Datasets
   emissions_cleaned.csv (7,630 rows)


   energy_cleaned.csv (7,696 rows)


   policies_cleaned.csv (13134 policies)


   covariates_merged.csv (7,630 rows)
   wb_indicators_cleaned.csv
   wgi_governance_cleaned.csv
   sector_emissions_cleaned.csv (8,228 rows)

 All cleaned datasets exported to: F:\Nexus_Projects\causal-climate-policy\data\interim
  Proceed to Notebook 03: Exploratory Data Analysis
